# Evaluate Model v2

Evaluate RT-DETR v2 models on the real test set.

**Model sources:**

- **W&B artifacts**: Load specific checkpoints (e.g. best mAP from a training round)
- **HF Hub**: Load published models from repo branches

**Metrics:**

1. mAP (COCO mAP@50:95, mAP@50, mAP@75)
2. Center-distance P/R/F1 at various distance thresholds
3. Confidence threshold sweep (P/R/F1 vs score threshold)
4. Visual prediction browsing

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.runs import list_wandb_model_artifacts, load_model_from_wandb
from moku.training import (
    collate_fn,
    evaluate_center_distance,
    evaluate_map,
    format_center_distance_results,
    make_eval_transform,
    sweep_confidence_threshold,
)
from moku.viz import plot_center_distance_comparison, plot_map_comparison, plot_threshold_sweep

/Users/hadim/Code/libs/moku/.pixi/envs/default/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


## Configuration

In [2]:
HF_DATASET = "kaya-go/moku-v2"
THRESHOLD = 0.05

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: mps


## Load Datasets


In [ ]:
# Synthetic dataset (stage 1 eval)
ds_synthetic = load_dataset(HF_DATASET, "synthetic")

# Real dataset (stage 2 eval)
ds_real = load_dataset(HF_DATASET, "real")

## Load Models from W&B

List available model artifacts and select which ones to evaluate.

In [ ]:
# List available model artifacts from W&B
artifacts_df = list_wandb_model_artifacts()
if not artifacts_df.empty:
    cols = [c for c in ["name", "version", "aliases", "run", "eval_map", "eval_map_50", "epoch", "size_mb"] if c in artifacts_df.columns]
    display(artifacts_df[cols].sort_values("eval_map", ascending=False).reset_index(drop=True))
else:
    print("No model artifacts found in W&B.")

Select artifacts to evaluate. Set the `ARTIFACTS` dict below (label → artifact name).

In [ ]:
# Set which artifacts to evaluate (label → W&B artifact name)
ARTIFACTS = {
    # "r4_lr5e-4_cos200": "model-r4_lr5e-4_cos200:latest",
    # "r4_lr1e-3_linear200": "model-r4_lr1e-3_linear200:latest",
}

# Load models
models = {}
for label, artifact_name in ARTIFACTS.items():
    print(f"Loading {label} from {artifact_name}...")
    ip, model = load_model_from_wandb(artifact_name)
    models[label] = {"ip": ip, "model": model}

print(f"\nLoaded {len(models)} models: {list(models.keys())}")

## mAP on Real Test

In [ ]:
metrics_map = {}

for label, entry in models.items():
    ds_real.set_transform(make_eval_transform(entry["ip"]))
    metrics_map[label] = evaluate_map(
        model=entry["model"],
        dataset=ds_real["test"],
        image_processor=entry["ip"],
        batch_size=8,
        threshold=THRESHOLD,
    )

if metrics_map:
    plot_map_comparison(metrics_map, title="mAP — Real Test")
else:
    print("No models loaded — nothing to evaluate.")

## Center-Distance Evaluation

mAP penalizes bbox size mismatches, but for our use case only the **center point** matters:

- **board_corner**: center → homography point
- **stones**: center → snap to nearest grid intersection

The center-distance metric reports Precision/Recall/F1 at various distance thresholds
(as % of image diagonal). A detection is "correct" if its center is within X% of the
diagonal from the nearest GT center of the same class.

In [ ]:
cd_results = {}

for label, entry in models.items():
    ds_real.set_transform(make_eval_transform(entry["ip"]))
    cd_results[label] = evaluate_center_distance(
        model=entry["model"],
        dataset=ds_real["test"],
        image_processor=entry["ip"],
        batch_size=8,
        threshold=THRESHOLD,
    )

if cd_results:
    plot_center_distance_comparison(cd_results, title="Center Distance — Real Test")
else:
    print("No models loaded — nothing to evaluate.")

## Summary Table

In [ ]:
# Combine mAP and center-distance into a summary table
summary_rows = []
for label in models:
    row = {"model": label}
    if label in metrics_map:
        m = metrics_map[label]
        row["mAP"] = m.get("map", float("nan"))
        row["mAP_50"] = m.get("map_50", float("nan"))
        row["mAP_75"] = m.get("map_75", float("nan"))
    if label in cd_results:
        cd = cd_results[label]
        # Macro F1 at 2% threshold
        f1s = [cd["per_class"][c]["thresholds"]["2%"]["f1"] for c in ID_TO_CATEGORY.values()]
        row["cd_F1@2%"] = np.mean(f1s)
    summary_rows.append(row)

if summary_rows:
    summary_df = pd.DataFrame(summary_rows).sort_values("mAP", ascending=False).reset_index(drop=True)
    display(summary_df)
else:
    print("No results to summarize.")

## Confidence Threshold Sweep

Find the optimal confidence threshold by sweeping P/R/F1 (center-distance @2%) vs score threshold.
Inference runs only once per model — threshold filtering is applied post-hoc.

In [ ]:
# Pick the best model (highest macro F1@2% center-distance)
if cd_results:
    best_label = max(
        cd_results,
        key=lambda l: np.mean([cd_results[l]["per_class"][c]["thresholds"]["2%"]["f1"] for c in ID_TO_CATEGORY.values()]),
    )
    best_entry = models[best_label]
    print(f"Sweeping confidence threshold for best model: {best_label}")

    ds_real.set_transform(make_eval_transform(best_entry["ip"]))
    sweep = sweep_confidence_threshold(
        model=best_entry["model"],
        dataset=ds_real["test"],
        image_processor=best_entry["ip"],
        batch_size=8,
        distance_threshold=0.02,
    )
    plot_threshold_sweep(sweep, title=f"Threshold Sweep — {best_label}")
else:
    print("No center-distance results — run the previous cell first.")

## Browse Predictions

Interactive widget to browse model predictions on dataset samples.


In [ ]:
from moku.viz import browse_predictions

# Build model registry for browsing
all_models = {}
for label, entry in models.items():
    all_models[label] = {"model": entry["model"].to(device), "image_processor": entry["ip"]}

# Build dataset registry
ds_synthetic = load_dataset(HF_DATASET, "synthetic")
all_datasets = {"Real": ds_real, "Synthetic": ds_synthetic}

browse_predictions(all_models, all_datasets, threshold=THRESHOLD)